
## Stochastic gradient descent

The Gabor model has two parameters, so we could find the global minimum by either (i)
exhaustively searching the parameter space or (ii) repeatedly starting gradient descent
from different positions and choosing the result with the lowest loss. However, neural
network models can have millions of parameters, so neither approach is practical. In
short, using gradient descent to find the global optimum of a high-dimensional loss
function is challenging. We can find a minimum, but there is no way to tell whether this is the global minimum or even a good one.
One of the main problems is that the final destination of a gradient descent algorithm is entirely determined by the starting point. Stochastic gradient descent (SGD) attempts
to remedy this problem by adding some noise to the gradient at each step. The solution
still moves downhill on average, but at any given iteration, the direction chosen is not
necessarily in the steepest downhill direction. Indeed, it might not be downhill at all.
The SGD algorithm has the possibility of moving temporarily uphill and hence jumping
from one “valley” of the loss function to another (figure 6.5b).


<img src="assets/Chap06/TrainGaborGDSGD.svg" style="filter: invert(1);" width="100%">

Gradient descent vs. stochastic gradient descent. a) Gradient descent
with line search. As long as the gradient descent algorithm is initialized in the
right “valley” of the loss function (e.g., points 1 and 3), the parameter estimate
will move steadily toward the global minimum. However, if it is initialized outside
this valley (e.g., point 2), it will descend toward one of the local minima. b)
Stochastic gradient descent adds noise to the optimization process, so it is possible
to escape from the wrong valley (e.g., point 2) and still reach the global minimum.


<img src="assets/Chap06/TrainGaborSGDIter.svg" style="filter: invert(1);" width="100%">

Alternative view of SGD for the Gabor model with a batch size three.
a) Loss function for the entire training dataset. At each iteration, there is a probability
distribution of possible parameter changes (inset shows samples). These
correspond to different choices of the three batch elements. b) Loss function for
one possible batch. The SGD algorithm moves in the downhill direction on this
function for a distance that is determined by the learning rate and the local gradient
magnitude. The current model (dashed function in inset) changes to better
fit the batch data (solid function). c) A different batch creates a different loss
function and results in a different update. d) For this batch, the algorithm moves
downhill with respect to the batch loss function but in the locally uphill direction
with respect to the global loss function in panel (a). This is how SGD can escape
local minima.

### Batches and epochs

The mechanism for introducing randomness is simple. At each iteration, the algorithm
chooses a random subset of the training data and computes the gradient from these
examples alone. This subset is known as a minibatch or batch for short. The update rule
for the model parameters ϕt at iteration t is hence:

\begin{eqnarray}\label{eq:train_sgd}
  \boldsymbol\phi_{t+1}\longleftarrow\boldsymbol\phi_{t} - \alpha \cdot \sum_{i\in\mathcal{B}_{t}}\frac{\partial \ell_{i}[\boldsymbol\phi_{t}]}{\partial \boldsymbol\phi},
 \end{eqnarray}

where Bt is a set containing the indices of the input/output pairs in the current batch
and, as before, ℓi is the loss due to the ith pair. The term α is the learning rate, and
together with the gradient magnitude, determines the distance moved at each iteration.
The learning rate is chosen at the start of the procedure and does not depend on the
local properties of the function.
The batches are usually drawn from the dataset without replacement. The algorithm
works through the training examples until it has used all the data, at which point it starts sampling from the full training dataset again. A single pass through the entire
training dataset is referred to as an epoch. A batch may be as small as a single example
or as large as the whole dataset. The latter case is called full-batch gradient descent and
is identical to regular (non-stochastic) gradient descent.
An alternative interpretation of SGD is that it computes the gradient of a different
loss function at each iteration; the loss function depends on both the model and the
training data and hence will differ for each randomly selected batch. In this view,
SGD performs deterministic gradient descent on a constantly changing loss function
(figure 6.6). However, despite this variability, the expected loss and expected gradients
at any point remain the same as for gradient descent.

### Properties of stochastic gradient descent


SGD has several attractive features. First, although it adds noise to the trajectory, it
still improves the fit to a subset of the data at each iteration. Hence, the updates tend
to be sensible even if they are not optimal. Second, because it draws training examples
without replacement and iterates through the dataset, the training examples all still
contribute equally. Third, it is less computationally expensive to compute the gradient from just a subset of the training data. Fourth, it can (in principle) escape local minima.
Fifth, it reduces the chances of getting stuck near saddle points; it is likely that at least
some of the possible batches will have a significant gradient at any point on the loss
function. Finally, there is some evidence that SGD finds parameters for neural networks
that cause them to generalize well to new data in practice (see section 9.2).
SGD does not necessarily “converge” in the traditional sense. However, the hope is
that when we are close to the global minimum, all the data points will be well described
by the model. Consequently, the gradient will be small, whichever batch is chosen, and
the parameters will cease to change much. In practice, SGD is often applied with a
learning rate schedule. The learning rate α starts at a high value and is decreased by a
constant factor every N epochs. The logic is that in the early stages of training, we want
the algorithm to explore the parameter space, jumping from valley to valley to find a
sensible region. In later stages, we are roughly in the right place and are more concerned
with fine-tuning the parameters, so we decrease α to make smaller changes.


In [ ]:
# TODO Experiment with starting the optimization in the previous cell in different places
# and show that it heads to a local minimum if we don't start it in the right valley
     

def gradient_descent_step_fixed_learning_rate(phi, data, alpha):
  # TODO -- fill in this routine so that we take a fixed size step of size alpha without using line search

  return phi
     

# Initialize the parameters
n_steps = 21
phi_all = np.zeros((2,n_steps+1))
phi_all[0,0] = -1.5
phi_all[1,0] = 8.5

# Measure loss and draw initial model
loss =  compute_loss(data[0,:], data[1,:], model, phi_all[:,0:1])
draw_model(data,model,phi_all[:,0:1], "Initial parameters, Loss = %f"%(loss))

for c_step in range (n_steps):
  # Do gradient descent step
  phi_all[:,c_step+1:c_step+2] = gradient_descent_step_fixed_learning_rate(phi_all[:,c_step:c_step+1],data, alpha =0.2)
  # Measure loss and draw model every 4th step
  if c_step % 4 == 0:
    loss =  compute_loss(data[0,:], data[1,:], model, phi_all[:,c_step+1:c_step+2])
    draw_model(data,model,phi_all[:,c_step+1], "Iteration %d, loss = %f"%(c_step+1,loss))

draw_loss_function(compute_loss, data, model,phi_all)

     

# TODO Experiment with the learning rate, alpha.
# What happens if you set it too large?
# What happens if you set it too small?
     

def stochastic_gradient_descent_step(phi, data, alpha, batch_size):
  # TODO -- fill in this routine so that we take a fixed size step of size alpha but only using a subset (batch) of the data
  # at each step
  # You can use the function np.random.permutation to generate a random permutation of the n_data = data.shape[1] indices
  # and then just choose the first n=batch_size of these indices.  Then compute the gradient update
  # from just the data with these indices.   More properly, you should sample without replacement, but this will do for now.


  return phi
     

# Set the random number generator so you always get same numbers (disable if you don't want this)
np.random.seed(1)
# Initialize the parameters
n_steps = 41
phi_all = np.zeros((2,n_steps+1))
phi_all[0,0] = 3.5
phi_all[1,0] = 6.5

# Measure loss and draw initial model
loss =  compute_loss(data[0,:], data[1,:], model, phi_all[:,0:1])
draw_model(data,model,phi_all[:,0:1], "Initial parameters, Loss = %f"%(loss))

for c_step in range (n_steps):
  # Do gradient descent step
  phi_all[:,c_step+1:c_step+2] = stochastic_gradient_descent_step(phi_all[:,c_step:c_step+1],data, alpha =0.8, batch_size=5)
  # Measure loss and draw model every 8th step
  if c_step % 8 == 0:
    loss =  compute_loss(data[0,:], data[1,:], model, phi_all[:,c_step+1:c_step+2])
    draw_model(data,model,phi_all[:,c_step+1], "Iteration %d, loss = %f"%(c_step+1,loss))

draw_loss_function(compute_loss, data, model,phi_all)
     

# TODO -- Experiment with different learning rates, starting points, batch sizes, number of steps.  Get a feel for this.
     

# TODO -- Add a learning rate schedule.  Reduce the learning rate by a factor of beta every M iterations